# Using the Panoptes Aggregation Tool from Zooniverse
See Readme for the installation instructions.


This notebook combines the aggregation steps from the raw classification from zooniverse.
Installed from github, https://aggregation-caesar.zooniverse.org/Scripts.html#scripts
The latest pypi release is from 2020 and I can't install it: https://pypi.org/project/panoptes-aggregation/


In [106]:
# install the aggregation tool
!pip install -U git+https://github.com/zooniverse/aggregation-for-caesar.git

  Cloning https://github.com/zooniverse/aggregation-for-caesar.git to /private/var/folders/2k/78nn7s4548986wsjh29rhj9w0000gn/T/pip-req-build-aw1mgm31
  Running command git clone --filter=blob:none --quiet https://github.com/zooniverse/aggregation-for-caesar.git /private/var/folders/2k/78nn7s4548986wsjh29rhj9w0000gn/T/pip-req-build-aw1mgm31
  Resolved https://github.com/zooniverse/aggregation-for-caesar.git to commit c8a9fffb5228b962515eecd96e852ae0aebaa681
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for panoptes_aggregation: filename=panoptes_aggregation-4.1.0-py3-none-any.whl size=281363 sha256=af22c85ac17add5e2d3ae047cd6b730b7a7a743c1f4d4e209a3a06bdb905aa00
  Stored in directory: /private/var/folders/2k/78nn7s4548986wsjh29rhj9w0000gn/T/pip-ephem-wheel-cache-4lxn_l3p/wheels/94/2e/ea/df4a44af95cb9ea32bcb3912735314cbb57c5a5c59031e903f
Succ

In [101]:
!pip uninstall -y panoptes_aggregation

Found existing installation: panoptes_aggregation 4.1.0
Uninstalling panoptes_aggregation-4.1.0:
  Successfully uninstalled panoptes_aggregation-4.1.0


In [103]:
!pip install panoptes_aggregation

  Using cached panoptes_aggregation-4.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached beautifulsoup4-4.10.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached collatex-2.2-py2.py3-none-any.whl.metadata (10 kB)
  Using cached hdbscan-0.8.28.tar.gz (5.2 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-1.23.1.tar.gz (10.7 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached packaging-21.3-py3-none-any.whl.metadata (15 kB)
  Using cached pandas-1.4.3.tar.gz (4.9 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached progressbar2-4.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached python-Levenshtein-0.12.2.tar.gz (50 kB)
  Preparing metadata (setup.py) ... done
  Using cach

In [2]:
!pip install -r requirements-dev.txt

In [107]:
from zooniverse.config import get_config
import pandas as pd
from pathlib import Path

### use either the subset of the subset
#phase_tag = "Iguanas 1st launch"
#data_folder = "./data/phase_1"
## 

# phase_tag = "Iguanas 2nd launch"
# data_folder = "./data/phase_2"

phase_tag = "Iguanas 3rd launch"
data_folder = "./data/phase_3"

# phase_tag = "Iguanas 4th launch"
# data_folder = "./data/phase_4"

input_path = Path("/Users/christian/data/zooniverse")
# use_gold_standard_subset = "expert" # Use the expert-GS-Xphase as the basis
output_path = Path("/Users/christian/data/zooniverse/2024_09_02_analysis").joinpath(phase_tag).resolve()

workflow_id_p1 = 14370.0
workflow_id_p2 = 20600.0
workflow_id_p3 = 22040.0
workflow_id_p4 = 22600.0

output_plot_path = output_path.joinpath("plots")
output_plot_path.mkdir(parents=True, exist_ok=True)

reprocess = False

config = get_config(phase_tag=phase_tag, input_path=input_path, output_path=output_path)
config


KeyError: 'Iguanas 4th launch'

# Look into the subjects file
This contains the mappings from the subject_id to the image file

In [4]:
# read the original file
df_subjects = pd.read_csv("./data/zooniverse/iguanas-from-above-subjects.csv", sep=",")


In [108]:
# filter the subjects for only the images in the three phases

df_subjects = df_subjects[df_subjects.workflow_id.isin([workflow_id_p1, workflow_id_p2, workflow_id_p3, workflow_id_p4])]


In [6]:
# inspect the metadata
import json
def get_json_keys(json_str):
    try:
        json_obj = json.loads(json_str)
        return list(json_obj.keys())
    except json.JSONDecodeError:
        return []

# Apply the function to each row in the metadata column and collect all keys
all_keys = df_subjects['locations'].apply(get_json_keys)

# Flatten the list of lists and get unique keys
unique_keys = set([key for sublist in all_keys for key in sublist])

print(unique_keys)

{'0'}


Clean up the subjects file for inconsistent naming.

In [7]:
df_subjects["image_name"] = df_subjects['metadata'].apply(lambda x: json.loads(x).get('Image_name') 
                                        or json.loads(x).get('image_name') 
                                        or json.loads(x).get('Filename')).sort_values(ascending=True)

# 'site', 'flight', 'Flight', 'Site', 'flight_code' depict the same
df_subjects["flight_code"] = df_subjects['metadata'].apply(lambda x: json.loads(x).get('flight_code') 
                                        or json.loads(x).get('site') 
                                        or json.loads(x).get('flight')
                                        or json.loads(x).get('Flight')
                                        or json.loads(x).get('Site')).sort_values(ascending=True)

df_subjects["url"] = df_subjects['locations'].apply(lambda x: json.loads(x)["0"])
df_subjects["filepath"] = None

In [8]:
df_subjects

,subject_id,project_id,workflow_id,subject_set_id,metadata,locations,classifications_count,retired_at,retirement_reason,created_at,updated_at,image_name,flight_code,url,filepath
190,47967468,11905,14370.0,86008,"{""site"":""SFB"",""image_name"":""SFB01-3_08.jpg"",""s...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2020-11-15 19:06:16 UTC,classification_count,2020-07-18 20:38:14 UTC,2020-07-18 20:38:14 UTC,SFB01-3_08.jpg,SFB,https://panoptes-uploads.zooniverse.org/subjec...,None
191,47967469,11905,14370.0,86008,"{""site"":""SFB"",""image_name"":""SFB01-3_15.jpg"",""s...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2020-10-28 19:25:18 UTC,classification_count,2020-07-18 20:38:17 UTC,2020-07-18 20:38:17 UTC,SFB01-3_15.jpg,SFB,https://panoptes-uploads.zooniverse.org/subjec...,None
192,47967470,11905,14370.0,86008,"{""site"":""SFB"",""image_name"":""SFB01-3_27.jpg"",""s...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2020-11-14 10:07:19 UTC,classification_count,2020-07-18 20:38:18 UTC,2020-07-18 20:38:18 UTC,SFB01-3_27.jpg,SFB,https://panoptes-uploads.zooniverse.org/subjec...,None
193,47967471,11905,14370.0,86008,"{""site"":""SFB"",""image_name"":""SFB01-3_28.jpg"",""s...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2020-11-09 10:36:02 UTC,classification_count,2020-07-18 20:38:20 UTC,2020-07-18 20:38:20 UTC,SFB01-3_28.jpg,SFB,https://panoptes-uploads.zooniverse.org/subjec...,None
194,47967472,11905,14370.0,86008,"{""site"":""SFB"",""image_name"":""SFB01-3_34.jpg"",""s...","{""0"":""https://panoptes-uploads.zooniverse.org/...",20,2020-11-18 20:44:36 UTC,classification_count,2020-07-18 20:38:22 UTC,2020-07-18 20:38:22 UTC,SFB01-3_34.jpg,SFB,https://panoptes-uploads.zooniverse.org/subjec...,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58027,78965182,11905,22040.0,106640,"{""id"":""476"",""set"":""SouthCoastH"",""site"":""SouthC...","{""0"":""https://panoptes-uploads.zooniverse.org/...",30,2023-06-04 20:04:00 UTC,classification_count,2022-07-24 11:23:51 UTC,2022-07-24 11:23:51 UTC,ESCH02-2_80.jpg,SouthCoastH,https://panoptes-uploads.zooniverse.org/subjec...,None
58028,78965183,11905,22040.0,106640,"{""id"":""477"",""set"":""SouthCoastH"",""site"":""SouthC...","{""0"":""https://panoptes-uploads.zooniverse.org/...",24,2023-08-30 16:43:06 UTC,classification_count,2022-07-24 11:23:51 UTC,2023-07-28 07:13:45 UTC,ESCH02-2_81.jpg,SouthCoastH,https://panoptes-uploads.zooniverse.org/subjec...,None
58029,78965184,11905,22040.0,106640,"{""id"":""478"",""set"":""SouthCoastH"",""site"":""SouthC...","{""0"":""https://panoptes-uploads.zooniverse.org/...",30,2023-07-14 13:55:55 UTC,classification_count,2022-07-24 11:23:52 UTC,2022-07-24 11:23:52 UTC,ESCH02-2_91.jpg,SouthCoastH,https://panoptes-uploads.zooniverse.org/subjec...,None
58030,78965185,11905,22040.0,106640,"{""id"":""479"",""set"":""SouthCoastH"",""site"":""SouthC...","{""0"":""https://panoptes-uploads.zooniverse.org/...",30,2023-07-20 04:37:58 UTC,classification_count,2022-07-24 11:23:53 UTC,2022-07-24 11:23:53 UTC,ESCH02-2_92.jpg,SouthCoastH,https://panoptes-uploads.zooniverse.org/subjec...,None


helper function to download the images using the urls in the subjects file

In [9]:
from loguru import logger
from time import sleep

import requests

def download_image(url, filename):
    """Download an image from a URL and save it to a file."""
    try:
        response = requests.get(url)
        if response.status_code == 200:
            with open(filename, 'wb') as file:
                file.write(response.content)
            return True
        else:
            logger.warning(f"Failed to download {url}")
            logger.error(response)
            sleep(5)
            return False
    except Exception as e:
        logger.error(e)
        sleep(5)
        return False

# Panoptes Data Extraction from Zooniverse
## Panoptes config
### Create the configuration files automatically
The configurations were changed to custom workflow versions.

In [112]:
# create a configuration file from the workflow
#!mkdir ./data/phase_1
#! panoptes_aggregation config /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-workflows.csv 14370 --min_version 0 --max_version 142.245 -d ./data/phase_1
# 
#!mkdir ./data/phase_2
#! panoptes_aggregation config /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-workflows.csv 20600 --min_version 0 --max_version 94.166 -d ./data/phase_2
# 
#!mkdir ./data/phase_3
#! panoptes_aggregation config /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-workflows.csv 22040 --min_version 0 --max_version 9.63 -d ./data/phase_3

!mkdir ./data/phase_4
! panoptes_aggregation config /Users/christian/data/zooniverse/IguanasFromAbove/2024-08-08/iguanas-from-above-workflows.csv 22600 -v 94.166 -d ./data/phase_4


mkdir: ./data/phase_4: File exists
Traceback (most recent call last):
  File "/Users/christian/opt/anaconda3/envs/iguanas-from-above-zooniverse_311/bin/panoptes_aggregation", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/Users/christian/opt/anaconda3/envs/iguanas-from-above-zooniverse_311/lib/python3.11/site-packages/panoptes_aggregation/scripts/aggregation_parser.py", line 271, in main
    panoptes_aggregation.scripts.config_workflow(
  File "/Users/christian/opt/anaconda3/envs/iguanas-from-above-zooniverse_311/lib/python3.11/site-packages/panoptes_aggregation/scripts/config_workflow_panoptes.py", line 71, in config_workflow
    assert (wdx.sum() > 0), 'workflow ID and workflow version(s) combination does not exist'
            ^^^^^^^^^^^^^
AssertionError: workflow ID and workflow version(s) combination does not exist
Traceback (most recent call last):
  File "/Users/christian/opt/anaconda3/envs/iguanas-from-above-zooniverse_311/bin/panoptes_aggregation", line

## Have a look at the datasets

In [96]:
!tail /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv | grep workflow_translation_id

516322421,jenniferccguy,2484629,5df3221c2868fbefce6d,25379,Plastics GS dataset,15.25,2023-10-15 18:36:12 UTC,,,"{""source"":""api"",""session"":""8327675098ea28ef4ed9f73d92d97a6af828eae69368ee3880c555e7ddfb0b10"",""viewport"":{""width"":1440,""height"":783},""started_at"":""2023-10-15T18:36:02.119Z"",""user_agent"":""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"",""utc_offset"":""-3600"",""finished_at"":""2023-10-15T18:36:12.785Z"",""live_project"":true,""interventions"":{""opt_in"":true,""messageShown"":false},""user_language"":""en"",""user_group_ids"":[],""subject_dimensions"":[{""clientWidth"":720,""clientHeight"":705,""naturalWidth"":1053,""naturalHeight"":1030}],""subject_selection_state"":{""retired"":false,""selected_at"":""2023-10-15T18:35:11.684Z"",""already_seen"":false,""selection_state"":""normal"",""finished_workflow"":false,""user_has_finished_workflow"":false},""workflow_translation_id"":""68195""}

In [93]:
!tail /Users/christian/data/zooniverse/IguanasFromAbove/2024-08-08/iguanas-from-above-classifications.csv

570348141,Ekrouse,2169311,dc443c2a3eb398451a07,25351,Iguanas 4th launch,51.133,2024-06-30 19:32:11 UTC,,,"{""source"":""api"",""session"":""e7afd21d7e2fd8b501f25dfef7aaa452335325c66153fb356d495f73849b4904"",""viewport"":{""width"":1138,""height"":640},""started_at"":""2024-06-29T23:15:41.205Z"",""user_agent"":""Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"",""utc_offset"":""14400"",""finished_at"":""2024-06-30T19:32:11.454Z"",""live_project"":true,""interventions"":{""opt_in"":true,""messageShown"":false},""user_language"":""en"",""user_group_ids"":[],""subject_dimensions"":[{""clientWidth"":580,""clientHeight"":576,""naturalWidth"":719,""naturalHeight"":714}],""subject_selection_state"":{""retired"":false,""selected_at"":""2024-06-29T23:15:13.814Z"",""already_seen"":false,""selection_state"":""normal"",""finished_workflow"":false,""user_has_finished_workflow"":false},""workflow_translation_id"":""68096""}","[{""task"":""T0"",""t

## Extract the data

In [11]:
# phase 1
if data_folder == "./data/phase_1":
    !mkdir ./data/phase_1/V121.144
    !mkdir ./data/phase_1/V134.236
    
    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_1/V121.144 ./data/phase_1/Extractor_config_workflow_14370_V121.144.yaml
    
    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_1/V134.236 ./data/phase_1/Extractor_config_workflow_14370_V134.236-1.yaml
    
else:
    print(f"No data to process because of the data_folder: {data_folder}")


No data to process because of the data_folder: ./data/phase_3


In [12]:
if data_folder == "./data/phase_2" and reprocess == True:
    # phase 2
    
    !mkdir ./data/phase_2/V89.162
    !mkdir ./data/phase_2/V93.166
    !mkdir ./data/phase_2/V94.166 
    
    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_2/V89.162 ./data/phase_2/Extractor_config_workflow_20600_V89.162.yaml
    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_2/V93.166 ./data/phase_2/Extractor_config_workflow_20600_V93.166.yaml
    !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_2/V94.166 ./data/phase_2/Extractor_config_workflow_20600_V94.166.yaml

else:
    print(f"No data to process because of the data_folder: {data_folder}")

No data to process because of the data_folder: ./data/phase_3


## Extacting data based on the classifications from 2023-10-15


In [13]:
# if data_folder == "./data/phase_3":
#     !mkdir ./data/phase_3/V7.63    
#     !mkdir ./data/phase_3/V9.63
#     
#     !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_3/V7.63 ./data/phase_3/Extractor_config_workflow_22040_V7.63.yaml
#     !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2023-10-15/iguanas-from-above-classifications.csv -d ./data/phase_3/V9.63 ./data/phase_3/Extractor_config_workflow_22040_V9.63.yaml
#     
# else:
#     print(f"No data to process because of the data_folder: {data_folder}")

## Extacting data based on the newer classifications from 2024-08-08

In [14]:
# if data_folder == "./data/phase_3":
#     !mkdir ./data/phase_3/V7.63_08-08
#     !mkdir ./data/phase_3/V9.63_08-08
#     
#     !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2024-08-08/iguanas-from-above-classifications.csv -d ./data/phase_3/V7.63_08-08 ./data/phase_3/Extractor_config_workflow_22040_V7.63.yaml
#     !panoptes_aggregation extract /Users/christian/data/zooniverse/IguanasFromAbove/2024-08-08/iguanas-from-above-classifications.csv -d ./data/phase_3/V9.63_08-08 ./data/phase_3/Extractor_config_workflow_22040_V9.63.yaml
#     
# else:
#     print(f"No data to process because of the data_folder: {data_folder}")

### Merge the single point and questions extractions

In [15]:
# phase 1
if data_folder == "./data/phase_1":
    df_panoptes_point_extractor_1 = pd.read_csv(f"./data/phase_1/V121.144/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_2 = pd.read_csv(f"./data/phase_1/V134.236/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_1["workflow_version"] = "121.144"
    df_panoptes_point_extractor_2["workflow_version"] = "134.236"
    
    df_panoptes_question_1 = pd.read_csv(f"{data_folder}/V121.144/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_2 = pd.read_csv(f"{data_folder}/V134.236/question_extractor_extractions.csv", sep=",")
    
    df_panoptes_point_extractor = pd.concat([df_panoptes_point_extractor_1, df_panoptes_point_extractor_2], axis=0)
    df_panoptes_question = pd.concat([df_panoptes_question_1, df_panoptes_question_2], axis=0)
    
    df_panoptes_point_extractor
    
else:
    print(f"No data to process because of the data_folder: {data_folder}")

No data to process because of the data_folder: ./data/phase_3


In [16]:
# # phase 2
if data_folder == "./data/phase_2":
    # read the rectangles annotations too there
    df_panotes_rectangle_extractor_1 = pd.read_csv(f"{data_folder}/V89.162/shape_extractor_rectangle_extractions.csv", sep=",")
    
    df_panoptes_point_extractor_1 = pd.read_csv(f"{data_folder}/V89.162/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_2 = pd.read_csv(f"{data_folder}/V93.166/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_3 = pd.read_csv(f"{data_folder}/V94.166/point_extractor_by_frame_extractions.csv", sep=",")
    
    df_panoptes_point_extractor_1["workflow_version"] = "89.162"
    df_panoptes_point_extractor_2["workflow_version"] = "93.166"
    df_panoptes_point_extractor_3["workflow_version"] = "94.166"
    
    df_panoptes_question_1 = pd.read_csv(f"{data_folder}/V89.162/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_2 = pd.read_csv(f"{data_folder}/V93.166/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_3 = pd.read_csv(f"{data_folder}/V94.166/question_extractor_extractions.csv", sep=",")
    
    df_panoptes_point_extractor = pd.concat([df_panoptes_point_extractor_1, df_panoptes_point_extractor_2, df_panoptes_point_extractor_3], axis=0)
    df_panoptes_question = pd.concat([df_panoptes_question_1, df_panoptes_question_2, df_panoptes_question_3], axis=0)

    df_panotes_rectangle_extractor_1
    
else:
    print(f"No data to process because of the data_folder: {data_folder}")

No data to process because of the data_folder: ./data/phase_3


### 2023 data

In [41]:
if data_folder == "./data/phase_3":
    df_panoptes_point_extractor_1 = pd.read_csv(f"{data_folder}/V7.63/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_2 = pd.read_csv(f"{data_folder}/V9.63/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_1["workflow_version"] = "7.63"
    df_panoptes_point_extractor_2["workflow_version"] = "9.63"

    df_panoptes_question_1 = pd.read_csv(f"{data_folder}/V7.63/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_2 = pd.read_csv(f"{data_folder}/V9.63/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_1["workflow_version"] = "7.63"
    df_panoptes_question_2["workflow_version"] = "9.63"

    df_panoptes_point_extractor = pd.concat([df_panoptes_point_extractor_1, df_panoptes_point_extractor_2], axis=0)
    df_panoptes_question = pd.concat([df_panoptes_question_1, df_panoptes_question_2], axis=0)
    
else:
    print(f"No data to process because of the data_folder: {data_folder}")


In [18]:
df_panoptes_point_extractor.drop(columns=["user_name", "user_id"], inplace=False)

,classification_id,workflow_id,task,created_at,subject_id,extractor,data.aggregation_version,data.frame0.T2_tool1_x,data.frame0.T2_tool1_y,data.frame0.T4_tool0_x,...,data.frame0.T4_tool7_y,data.frame0.T2_tool3_x,data.frame0.T2_tool3_y,data.frame0.T4_tool5_x,data.frame0.T4_tool5_y,data.frame0.T4_tool6_x,data.frame0.T4_tool6_y,data.frame0.T4_tool4_x,data.frame0.T4_tool4_y,workflow_version
0,428441443,22040,T2,2022-07-22 14:32:53 UTC,78861918,point_extractor_by_frame,4.1.0,[557.1560668945312],[920.5582885742188],NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63
1,428441443,22040,T4,2022-07-22 14:32:53 UTC,78861918,point_extractor_by_frame,4.1.0,NaN,NaN,"[1035.0341796875, 623.5692138671875, 497.99880...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63
2,428463469,22040,T2,2022-07-22 16:29:00 UTC,78861913,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63
3,428463469,22040,T4,2022-07-22 16:29:00 UTC,78861913,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63
4,428803453,22040,T2,2022-07-25 08:44:40 UTC,78961556,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1425241,511354194,22040,T4,2023-09-16 23:41:20 UTC,78922610,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63
1425242,511354201,22040,T2,2023-09-16 23:41:27 UTC,78922599,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63
1425243,511354201,22040,T4,2023-09-16 23:41:27 UTC,78922599,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63
1425244,511354219,22040,T2,2023-09-16 23:41:43 UTC,78922611,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63


In [ ]:
df_panoptes_point_extractor

### 2024 data

In [71]:
if data_folder == "./data/phase_3":
    df_panoptes_point_extractor_1 = pd.read_csv(f"{data_folder}/V7.63_08-08/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_2 = pd.read_csv(f"{data_folder}/V9.63_08-08/point_extractor_by_frame_extractions.csv", sep=",")
    df_panoptes_point_extractor_1["workflow_version"] = "7.63"
    df_panoptes_point_extractor_2["workflow_version"] = "9.63"

    df_panoptes_question_1 = pd.read_csv(f"{data_folder}/V7.63/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_2 = pd.read_csv(f"{data_folder}/V9.63/question_extractor_extractions.csv", sep=",")
    df_panoptes_question_1["workflow_version"] = "7.63"
    df_panoptes_question_2["workflow_version"] = "9.63"

    df_panoptes_point_extractor = pd.concat([df_panoptes_point_extractor_1, df_panoptes_point_extractor_2], axis=0)
    df_panoptes_question = pd.concat([df_panoptes_question_1, df_panoptes_question_2], axis=0)
    
else:
    print(f"No data to process because of the data_folder: {data_folder}")

In [72]:
df_panoptes_point_extractor.drop(columns=["user_name", "user_id"], inplace=False)

,classification_id,workflow_id,task,created_at,subject_id,extractor,data.aggregation_version,data.frame0.T2_tool1_x,data.frame0.T2_tool1_y,data.frame0.T4_tool0_x,...,data.frame0.T4_tool7_y,data.frame0.T2_tool3_x,data.frame0.T2_tool3_y,data.frame0.T4_tool5_x,data.frame0.T4_tool5_y,data.frame0.T4_tool6_x,data.frame0.T4_tool6_y,data.frame0.T4_tool4_x,data.frame0.T4_tool4_y,workflow_version
0,428441443,22040,T2,2022-07-22 14:32:53 UTC,78861918,point_extractor_by_frame,4.1.0,[557.1560668945312],[920.5582885742188],NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63
1,428441443,22040,T4,2022-07-22 14:32:53 UTC,78861918,point_extractor_by_frame,4.1.0,NaN,NaN,"[1035.0341796875, 623.5692138671875, 497.99880...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63
2,428463469,22040,T2,2022-07-22 16:29:00 UTC,78861913,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63
3,428463469,22040,T4,2022-07-22 16:29:00 UTC,78861913,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63
4,428803453,22040,T2,2022-07-25 08:44:40 UTC,78961556,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1425245,511354219,22040,T4,2023-09-16 23:41:43 UTC,78922611,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63
1425246,535140531,22040,T2,2024-01-17 23:23:35 UTC,78922650,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63
1425247,535140531,22040,T4,2024-01-17 23:23:35 UTC,78922650,point_extractor_by_frame,4.1.0,NaN,NaN,[388.39166259765625],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63
1425248,535140532,22040,T2,2024-01-17 23:23:35 UTC,78922599,point_extractor_by_frame,4.1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63


## Merge the subjects file with the point extractor file to get the image name
This is necessary to get the image name for the points.

In [73]:
# join the image name from the subjects file
df_panoptes_point_extractor = df_panoptes_point_extractor.merge(df_subjects[["subject_id", "image_name"]], left_on="subject_id", right_on="subject_id")
df_panoptes_point_extractor = df_panoptes_point_extractor[df_panoptes_point_extractor.subject_id.isin(df_subjects.subject_id)]

df_panoptes_point_extractor

,classification_id,user_name,user_id,workflow_id,task,created_at,subject_id,extractor,data.aggregation_version,data.frame0.T2_tool1_x,...,data.frame0.T2_tool3_x,data.frame0.T2_tool3_y,data.frame0.T4_tool5_x,data.frame0.T4_tool5_y,data.frame0.T4_tool6_x,data.frame0.T4_tool6_y,data.frame0.T4_tool4_x,data.frame0.T4_tool4_y,workflow_version,image_name
0,428803453,Nomad_Purple,1312868.0,22040,T2,2022-07-25 08:44:40 UTC,78961556,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63,PCIE13-2-2_83.jpg
1,428803453,Nomad_Purple,1312868.0,22040,T4,2022-07-25 08:44:40 UTC,78961556,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63,PCIE13-2-2_83.jpg
2,430711757,kmorrisseyukyahoo.co.uk,1325881.0,22040,T2,2022-08-04 14:48:04 UTC,78961556,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,PCIE13-2-2_83.jpg
3,430711757,kmorrisseyukyahoo.co.uk,1325881.0,22040,T4,2022-08-04 14:48:04 UTC,78961556,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,PCIE13-2-2_83.jpg
4,432408154,Barti,2497726.0,22040,T2,2022-08-13 20:33:37 UTC,78961556,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,PCIE13-2-2_83.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1440569,506596846,not-logged-in-f6350ed5e6689ab653e7,NaN,22040,T4,2023-08-16 17:59:50 UTC,78923883,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,GWA01-1_180.jpg
1440570,506724429,not-logged-in-31ddcc1d99963e7c9960,NaN,22040,T2,2023-08-17 13:41:27 UTC,78923883,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,GWA01-1_180.jpg
1440571,506724429,not-logged-in-31ddcc1d99963e7c9960,NaN,22040,T4,2023-08-17 13:41:27 UTC,78923883,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,GWA01-1_180.jpg
1440572,506829613,not-logged-in-97b85c50671be4f91baa,NaN,22040,T2,2023-08-18 03:50:13 UTC,78923883,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,GWA01-1_180.jpg


## Anonymise the data

In [74]:
from hashlib import blake2b

df_panoptes_point_extractor["user_id"] = df_panoptes_point_extractor['user_id'].apply(lambda x: blake2b(str(x).encode(), digest_size=16).hexdigest() if not pd.isnull(x) else x)
# Anonymize 'user_name' by hashing
df_panoptes_point_extractor['user_name'] = df_panoptes_point_extractor['user_name'].apply(lambda x: blake2b(x.encode(), digest_size=16).hexdigest() if isinstance(x, str) else x)

df_panoptes_question["user_id"] = df_panoptes_question['user_id'].apply(lambda x: blake2b(str(x).encode(), digest_size=16).hexdigest() if not pd.isnull(x) else x)
# Anonymize 'user_name' by hashing
df_panoptes_question['user_name'] = df_panoptes_question['user_name'].apply(lambda x: blake2b(x.encode(), digest_size=16).hexdigest() if isinstance(x, str) else x)

In [75]:
df_panoptes_question[df_panoptes_question["data.yes"] == 1.0].groupby("subject_id").size().sort_values(ascending=False)

subject_id
78964280    31
78925099    31
78964277    31
78964722    31
78964794    31
            ..
78933830     1
78933831     1
78933836     1
78933837     1
78965186     1
Length: 12029, dtype: int64

## Determine the amount of yes Answers for "Is there an Iguana"

In [76]:
df_panoptes_question

,classification_id,user_name,user_id,workflow_id,task,created_at,subject_id,extractor,data.yes,data.aggregation_version,data.no,workflow_version
0,428441443,386fc0ec047b7e259744e72e8e64b9f9,ea57b1088a10fa7fef30ed0b344e2ca3,22040,T0,2022-07-22 14:32:53 UTC,78861918,question_extractor,1.0,4.1.0,NaN,7.63
1,428463469,386fc0ec047b7e259744e72e8e64b9f9,ea57b1088a10fa7fef30ed0b344e2ca3,22040,T0,2022-07-22 16:29:00 UTC,78861913,question_extractor,1.0,4.1.0,NaN,7.63
2,428803453,6d90c675de24df885cc880eab99a0cbe,5c78309a3a0e2fb4505af64d71e7a83d,22040,T0,2022-07-25 08:44:40 UTC,78961556,question_extractor,NaN,4.1.0,1.0,7.63
3,428804329,d881a54ceca557c3fcd4c41a779bdc79,8824af3b56d4acd5574a1a5e239d0348,22040,T0,2022-07-25 08:57:30 UTC,78939084,question_extractor,NaN,4.1.0,1.0,7.63
4,428830592,3e3f36ae551c3d0c2d21da11947419ac,fb780c1c46af055b562fae4f15c7207d,22040,T0,2022-07-25 13:30:14 UTC,78957387,question_extractor,NaN,4.1.0,1.0,7.63
...,...,...,...,...,...,...,...,...,...,...,...,...
712618,511354180,5db78ccea92857b77aeb4228f75e2030,e2b4fd6b6d39bebba0a2185fbc78edba,22040,T0,2023-09-16 23:41:11 UTC,78922581,question_extractor,NaN,4.1.0,1.0,9.63
712619,511354184,5db78ccea92857b77aeb4228f75e2030,e2b4fd6b6d39bebba0a2185fbc78edba,22040,T0,2023-09-16 23:41:14 UTC,78922582,question_extractor,NaN,4.1.0,1.0,9.63
712620,511354194,5db78ccea92857b77aeb4228f75e2030,e2b4fd6b6d39bebba0a2185fbc78edba,22040,T0,2023-09-16 23:41:20 UTC,78922610,question_extractor,NaN,4.1.0,1.0,9.63
712621,511354201,5db78ccea92857b77aeb4228f75e2030,e2b4fd6b6d39bebba0a2185fbc78edba,22040,T0,2023-09-16 23:41:27 UTC,78922599,question_extractor,NaN,4.1.0,1.0,9.63


In [77]:
df_panoptes_question_r = df_panoptes_question[df_panoptes_question.task == "T0"][["subject_id", "data.no", "data.yes"]].groupby("subject_id").sum()

df_panoptes_question_r = df_panoptes_question_r.reset_index()
df_panoptes_question_r = df_panoptes_question_r[df_panoptes_question_r.subject_id.isin(df_subjects.subject_id)]
df_panoptes_question_r

,subject_id,data.no,data.yes
2,78921848,24.0,0.0
3,78921849,31.0,0.0
4,78921850,29.0,2.0
5,78921851,30.0,1.0
6,78921852,31.0,0.0
...,...,...,...
24365,78965182,28.0,3.0
24366,78965183,5.0,20.0
24367,78965184,30.0,1.0
24368,78965185,13.0,18.0


In [78]:
df_panoptes_question_r.to_csv(output_path / config["panoptes_question"], index = False)

## Get the Point Marks Analysis Ready

Filter for T2 only

In [79]:
df_panoptes_point_extractor_r = df_panoptes_point_extractor[
    (df_panoptes_point_extractor.task == "T2")
]
df_panoptes_point_extractor_r.columns

Index(['classification_id', 'user_name', 'user_id', 'workflow_id', 'task',
       'created_at', 'subject_id', 'extractor', 'data.aggregation_version',
       'data.frame0.T2_tool1_x', 'data.frame0.T2_tool1_y',
       'data.frame0.T4_tool0_x', 'data.frame0.T4_tool0_y',
       'data.frame0.T2_tool0_x', 'data.frame0.T2_tool0_y',
       'data.frame0.T2_tool2_x', 'data.frame0.T2_tool2_y',
       'data.frame0.T4_tool3_x', 'data.frame0.T4_tool3_y',
       'data.frame0.T4_tool2_x', 'data.frame0.T4_tool2_y',
       'data.frame0.T4_tool1_x', 'data.frame0.T4_tool1_y',
       'data.frame0.T4_tool7_x', 'data.frame0.T4_tool7_y',
       'data.frame0.T2_tool3_x', 'data.frame0.T2_tool3_y',
       'data.frame0.T4_tool5_x', 'data.frame0.T4_tool5_y',
       'data.frame0.T4_tool6_x', 'data.frame0.T4_tool6_y',
       'data.frame0.T4_tool4_x', 'data.frame0.T4_tool4_y', 'workflow_version',
       'image_name'],
      dtype='object')

### Which tool is which now?
| Tool Name               | Classification                               |
|-------------------------|----------------------------------------------|
| data.frame0.T2_tool0_x  | Adult Male in a lek                          |
| data.frame0.T2_tool1_x  | Adult Male alone                             |
| data.frame0.T2_tool2_x  | Others (females, young males, juveniles)     |
| data.frame0.T2_tool3_x  | Partial iguana                               |
| data.frame0.T2_tool4_x  | Could be an iguana, not sure                 |

Is "Could be an iguana, not sure" and "Partial Iguana" are omitted.


In [80]:
df_panoptes_point_extractor_r

,classification_id,user_name,user_id,workflow_id,task,created_at,subject_id,extractor,data.aggregation_version,data.frame0.T2_tool1_x,...,data.frame0.T2_tool3_x,data.frame0.T2_tool3_y,data.frame0.T4_tool5_x,data.frame0.T4_tool5_y,data.frame0.T4_tool6_x,data.frame0.T4_tool6_y,data.frame0.T4_tool4_x,data.frame0.T4_tool4_y,workflow_version,image_name
0,428803453,6d90c675de24df885cc880eab99a0cbe,5c78309a3a0e2fb4505af64d71e7a83d,22040,T2,2022-07-25 08:44:40 UTC,78961556,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.63,PCIE13-2-2_83.jpg
2,430711757,c46159a74cc1f058c9cee2575138ae99,2b43a5981ef5e0c9345e32317105e429,22040,T2,2022-08-04 14:48:04 UTC,78961556,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,PCIE13-2-2_83.jpg
4,432408154,070b22b65ade1189e49fdd4074ffd19e,c902c528ae34dd70036e12f944e8dd28,22040,T2,2022-08-13 20:33:37 UTC,78961556,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,PCIE13-2-2_83.jpg
6,438063583,6be90f7c5906d966fa46278cab681c33,b88ed174577219f7bc20ed439b5b8349,22040,T2,2022-09-12 14:23:10 UTC,78961556,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,PCIE13-2-2_83.jpg
8,444573170,0814d5106f0eb5e30bef5e461b5bc509,56a1977f012ec126a5e97415b0e8b7c1,22040,T2,2022-10-14 23:11:19 UTC,78961556,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,PCIE13-2-2_83.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1440564,504670840,edf0fdcbbabe07f3ed93abf632c2e0c0,b59d03b689c8f933eed4ace8f7931639,22040,T2,2023-08-04 14:32:45 UTC,78923883,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,GWA01-1_180.jpg
1440566,505765122,4c7fe317496db725f9f7f44a9e7960fc,a9f82bdcd6b06c824928a4102a669c06,22040,T2,2023-08-11 02:32:51 UTC,78923883,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,GWA01-1_180.jpg
1440568,506596846,d8546b9840ce61c9f9825d0c16e0ccf1,NaN,22040,T2,2023-08-16 17:59:50 UTC,78923883,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,GWA01-1_180.jpg
1440570,506724429,5bee34ea2e62307313e81fed9dc59e89,NaN,22040,T2,2023-08-17 13:41:27 UTC,78923883,point_extractor_by_frame,4.1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.63,GWA01-1_180.jpg


In [81]:
# create a flat structure from the nested marks over multiple columns from that.
from ast import literal_eval


if data_folder == "./data/phase_3": 
    columns_keep_x = ['data.frame0.T2_tool0_x', 'data.frame0.T2_tool1_x', 'data.frame0.T2_tool2_x']
    columns_keep_y = ['data.frame0.T2_tool0_y', 'data.frame0.T2_tool1_y', 'data.frame0.T2_tool2_y']
else:
    columns_keep_x = ['data.frame0.T2_tool0_x', 'data.frame0.T2_tool1_x', 'data.frame0.T2_tool2_x', 'data.frame0.T2_tool4_x']
    columns_keep_y = ['data.frame0.T2_tool0_y', 'data.frame0.T2_tool1_y', 'data.frame0.T2_tool2_y', 'data.frame0.T2_tool4_y']

for col in columns_keep_x + columns_keep_y:
    df_panoptes_point_extractor_r[col] = df_panoptes_point_extractor_r[col].apply(lambda x: literal_eval(x) if pd.notnull(x) else [])

# Merge the lists in 'x' and 'y' coordinates
df_panoptes_point_extractor_r['x'] = df_panoptes_point_extractor_r[columns_keep_x].values.tolist()
df_panoptes_point_extractor_r['y'] = df_panoptes_point_extractor_r[columns_keep_y].values.tolist()

# Flatten the lists in each row for 'x' and 'y'
df_panoptes_point_extractor_r['x'] = df_panoptes_point_extractor_r['x'].apply(lambda x: [item for sublist in x for item in sublist])
df_panoptes_point_extractor_r['y'] = df_panoptes_point_extractor_r['y'].apply(lambda x: [item for sublist in x for item in sublist])

# Explode the DataFrame to separate rows for each x, y pair
# Explode the DataFrame based on these columns to get separate rows for each list element
df_panoptes_point_extractor_r

/var/folders/2k/78nn7s4548986wsjh29rhj9w0000gn/T/ipykernel_58827/3002241372.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_panoptes_point_extractor_r[col] = df_panoptes_point_extractor_r[col].apply(lambda x: literal_eval(x) if pd.notnull(x) else [])
/var/folders/2k/78nn7s4548986wsjh29rhj9w0000gn/T/ipykernel_58827/3002241372.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_panoptes_point_extractor_r['x'] = df_panoptes_point_extractor_r[columns_keep_x].values.tolist()
/var/folders/2k/78nn7s454

,classification_id,user_name,user_id,workflow_id,task,created_at,subject_id,extractor,data.aggregation_version,data.frame0.T2_tool1_x,...,data.frame0.T4_tool5_x,data.frame0.T4_tool5_y,data.frame0.T4_tool6_x,data.frame0.T4_tool6_y,data.frame0.T4_tool4_x,data.frame0.T4_tool4_y,workflow_version,image_name,x,y
0,428803453,6d90c675de24df885cc880eab99a0cbe,5c78309a3a0e2fb4505af64d71e7a83d,22040,T2,2022-07-25 08:44:40 UTC,78961556,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,NaN,NaN,NaN,NaN,7.63,PCIE13-2-2_83.jpg,[],[]
2,430711757,c46159a74cc1f058c9cee2575138ae99,2b43a5981ef5e0c9345e32317105e429,22040,T2,2022-08-04 14:48:04 UTC,78961556,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,NaN,NaN,NaN,NaN,9.63,PCIE13-2-2_83.jpg,[],[]
4,432408154,070b22b65ade1189e49fdd4074ffd19e,c902c528ae34dd70036e12f944e8dd28,22040,T2,2022-08-13 20:33:37 UTC,78961556,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,NaN,NaN,NaN,NaN,9.63,PCIE13-2-2_83.jpg,[],[]
6,438063583,6be90f7c5906d966fa46278cab681c33,b88ed174577219f7bc20ed439b5b8349,22040,T2,2022-09-12 14:23:10 UTC,78961556,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,NaN,NaN,NaN,NaN,9.63,PCIE13-2-2_83.jpg,[],[]
8,444573170,0814d5106f0eb5e30bef5e461b5bc509,56a1977f012ec126a5e97415b0e8b7c1,22040,T2,2022-10-14 23:11:19 UTC,78961556,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,NaN,NaN,NaN,NaN,9.63,PCIE13-2-2_83.jpg,[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1440564,504670840,edf0fdcbbabe07f3ed93abf632c2e0c0,b59d03b689c8f933eed4ace8f7931639,22040,T2,2023-08-04 14:32:45 UTC,78923883,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,NaN,NaN,NaN,NaN,9.63,GWA01-1_180.jpg,[],[]
1440566,505765122,4c7fe317496db725f9f7f44a9e7960fc,a9f82bdcd6b06c824928a4102a669c06,22040,T2,2023-08-11 02:32:51 UTC,78923883,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,NaN,NaN,NaN,NaN,9.63,GWA01-1_180.jpg,[],[]
1440568,506596846,d8546b9840ce61c9f9825d0c16e0ccf1,NaN,22040,T2,2023-08-16 17:59:50 UTC,78923883,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,NaN,NaN,NaN,NaN,9.63,GWA01-1_180.jpg,[],[]
1440570,506724429,5bee34ea2e62307313e81fed9dc59e89,NaN,22040,T2,2023-08-17 13:41:27 UTC,78923883,point_extractor_by_frame,4.1.0,[],...,NaN,NaN,NaN,NaN,NaN,NaN,9.63,GWA01-1_180.jpg,[],[]


In [82]:
df_panoptes_point_extractor_r = df_panoptes_point_extractor_r[
    ['classification_id', 'user_name', 'user_id', 'workflow_id',  'workflow_version', 'task',
     'created_at', 'subject_id', "image_name",
     'x', 'y'
     ]].reset_index(drop=True)

df_panoptes_point_extractor_r

,classification_id,user_name,user_id,workflow_id,workflow_version,task,created_at,subject_id,image_name,x,y
0,428803453,6d90c675de24df885cc880eab99a0cbe,5c78309a3a0e2fb4505af64d71e7a83d,22040,7.63,T2,2022-07-25 08:44:40 UTC,78961556,PCIE13-2-2_83.jpg,[],[]
1,430711757,c46159a74cc1f058c9cee2575138ae99,2b43a5981ef5e0c9345e32317105e429,22040,9.63,T2,2022-08-04 14:48:04 UTC,78961556,PCIE13-2-2_83.jpg,[],[]
2,432408154,070b22b65ade1189e49fdd4074ffd19e,c902c528ae34dd70036e12f944e8dd28,22040,9.63,T2,2022-08-13 20:33:37 UTC,78961556,PCIE13-2-2_83.jpg,[],[]
3,438063583,6be90f7c5906d966fa46278cab681c33,b88ed174577219f7bc20ed439b5b8349,22040,9.63,T2,2022-09-12 14:23:10 UTC,78961556,PCIE13-2-2_83.jpg,[],[]
4,444573170,0814d5106f0eb5e30bef5e461b5bc509,56a1977f012ec126a5e97415b0e8b7c1,22040,9.63,T2,2022-10-14 23:11:19 UTC,78961556,PCIE13-2-2_83.jpg,[],[]
...,...,...,...,...,...,...,...,...,...,...,...
720282,504670840,edf0fdcbbabe07f3ed93abf632c2e0c0,b59d03b689c8f933eed4ace8f7931639,22040,9.63,T2,2023-08-04 14:32:45 UTC,78923883,GWA01-1_180.jpg,[],[]
720283,505765122,4c7fe317496db725f9f7f44a9e7960fc,a9f82bdcd6b06c824928a4102a669c06,22040,9.63,T2,2023-08-11 02:32:51 UTC,78923883,GWA01-1_180.jpg,[],[]
720284,506596846,d8546b9840ce61c9f9825d0c16e0ccf1,NaN,22040,9.63,T2,2023-08-16 17:59:50 UTC,78923883,GWA01-1_180.jpg,[],[]
720285,506724429,5bee34ea2e62307313e81fed9dc59e89,NaN,22040,9.63,T2,2023-08-17 13:41:27 UTC,78923883,GWA01-1_180.jpg,[],[]


In [83]:
# explode the lists of marks per user into one row per mark
df_panoptes_point_extractor_r_ex = df_panoptes_point_extractor_r.apply(lambda x: x.explode() if x.name in ['x', 'y'] else x)

In [84]:
# images with no marks have NaN values in the 'merged_x' and 'merged_y' columns
df_panoptes_point_extractor_r_ex_dropped = df_panoptes_point_extractor_r_ex.dropna(subset=['x', 'y'], how='all').sort_values(by=['user_id', 'subject_id', 'task', 'created_at'])
df_panoptes_point_extractor_r_ex_dropped

,classification_id,user_name,user_id,workflow_id,workflow_version,task,created_at,subject_id,image_name,x,y
22676,430180580,096835c7b506ed9511c344f420d01f74,002400ef36f94c5e2a6ccc49859923d8,22040,7.63,T2,2022-08-01 20:06:32 UTC,78963883,FPA03_54.jpg,449.09314,399.687042
363195,501567924,9c249e3b4c1adc212dc055fabe626cfd,00346ebf6ae91002059d21fa7090e46b,22040,9.63,T2,2023-07-16 21:09:04 UTC,78925071,GWB01-1_218.jpg,511.535278,594.941162
363195,501567924,9c249e3b4c1adc212dc055fabe626cfd,00346ebf6ae91002059d21fa7090e46b,22040,9.63,T2,2023-07-16 21:09:04 UTC,78925071,GWB01-1_218.jpg,349.569031,711.556946
363195,501567924,9c249e3b4c1adc212dc055fabe626cfd,00346ebf6ae91002059d21fa7090e46b,22040,9.63,T2,2023-07-16 21:09:04 UTC,78925071,GWB01-1_218.jpg,255.988495,756.907471
363195,501567924,9c249e3b4c1adc212dc055fabe626cfd,00346ebf6ae91002059d21fa7090e46b,22040,9.63,T2,2023-07-16 21:09:04 UTC,78925071,GWB01-1_218.jpg,9.079877,245.093994
...,...,...,...,...,...,...,...,...,...,...,...
462760,494172012,93ab28acce8ac92d3b3036d3cac69de3,NaN,22040,9.63,T2,2023-06-06 12:35:40 UTC,78965185,ESCH02-2_92.jpg,267.777679,31.393913
462760,494172012,93ab28acce8ac92d3b3036d3cac69de3,NaN,22040,9.63,T2,2023-06-06 12:35:40 UTC,78965185,ESCH02-2_92.jpg,221.41127,52.645176
462760,494172012,93ab28acce8ac92d3b3036d3cac69de3,NaN,22040,9.63,T2,2023-06-06 12:35:40 UTC,78965185,ESCH02-2_92.jpg,198.228073,31.393913
462760,494172012,93ab28acce8ac92d3b3036d3cac69de3,NaN,22040,9.63,T2,2023-06-06 12:35:40 UTC,78965185,ESCH02-2_92.jpg,582.840454,152.431137


In [85]:
# cast x and y to int
df_panoptes_point_extractor_r_ex_dropped = df_panoptes_point_extractor_r_ex_dropped.astype({'x': 'int32', 'y': 'int32'})
df_panoptes_point_extractor_r_ex_dropped

,classification_id,user_name,user_id,workflow_id,workflow_version,task,created_at,subject_id,image_name,x,y
22676,430180580,096835c7b506ed9511c344f420d01f74,002400ef36f94c5e2a6ccc49859923d8,22040,7.63,T2,2022-08-01 20:06:32 UTC,78963883,FPA03_54.jpg,449,399
363195,501567924,9c249e3b4c1adc212dc055fabe626cfd,00346ebf6ae91002059d21fa7090e46b,22040,9.63,T2,2023-07-16 21:09:04 UTC,78925071,GWB01-1_218.jpg,511,594
363195,501567924,9c249e3b4c1adc212dc055fabe626cfd,00346ebf6ae91002059d21fa7090e46b,22040,9.63,T2,2023-07-16 21:09:04 UTC,78925071,GWB01-1_218.jpg,349,711
363195,501567924,9c249e3b4c1adc212dc055fabe626cfd,00346ebf6ae91002059d21fa7090e46b,22040,9.63,T2,2023-07-16 21:09:04 UTC,78925071,GWB01-1_218.jpg,255,756
363195,501567924,9c249e3b4c1adc212dc055fabe626cfd,00346ebf6ae91002059d21fa7090e46b,22040,9.63,T2,2023-07-16 21:09:04 UTC,78925071,GWB01-1_218.jpg,9,245
...,...,...,...,...,...,...,...,...,...,...,...
462760,494172012,93ab28acce8ac92d3b3036d3cac69de3,NaN,22040,9.63,T2,2023-06-06 12:35:40 UTC,78965185,ESCH02-2_92.jpg,267,31
462760,494172012,93ab28acce8ac92d3b3036d3cac69de3,NaN,22040,9.63,T2,2023-06-06 12:35:40 UTC,78965185,ESCH02-2_92.jpg,221,52
462760,494172012,93ab28acce8ac92d3b3036d3cac69de3,NaN,22040,9.63,T2,2023-06-06 12:35:40 UTC,78965185,ESCH02-2_92.jpg,198,31
462760,494172012,93ab28acce8ac92d3b3036d3cac69de3,NaN,22040,9.63,T2,2023-06-06 12:35:40 UTC,78965185,ESCH02-2_92.jpg,582,152


In [86]:
df_panoptes_point_extractor_r_ex_dropped.to_csv(config["flat_panoptes_points"], sep=",", index = False)

## Inspecting the results
Check the numbers for a single subject_id

In [87]:
### Looks the images in question

subject_id_2 = 72373250 
df_debug = df_panoptes_point_extractor_r_ex_dropped[(df_panoptes_point_extractor_r_ex_dropped.subject_id == subject_id_2)]
df_debug

,classification_id,user_name,user_id,workflow_id,workflow_version,task,created_at,subject_id,image_name,x,y


In [88]:
df_debug.groupby('user_name').size()


Series([], dtype: int64)

In [89]:
df_debug[df_debug.user_name == "CallieSanDiego"]

,classification_id,user_name,user_id,workflow_id,workflow_version,task,created_at,subject_id,image_name,x,y


## Download images
iguanas-from-above-subjects_with_url.csv will be used to track which url was already downlaoded.

In [38]:
## save the file the extra columns we need for downloading.
df_subjects.to_csv(output_path / "iguanas-from-above-subjects_with_url.csv")


# read the modified csv
df_subjects = pd.read_csv(output_path / "iguanas-from-above-subjects_with_url.csv")


In [39]:
# df_subjects = pd.read_csv(output_path / "iguanas-from-above-subjects_with_url.csv")

# downoaded_images_path = Path("./data/downloaded_images")
# downoaded_images_path.mkdir(exist_ok=True, parents=True)
# return_val = True
# # df = df_subjects[df_subjects.subject_id.isin([44660616, 47968406])]
# # df = df_subjects[df_subjects.subject_id.isin([44660616, 47968406])]
# for index, row in df_subjects[df_subjects.workflow_id.isin([workflow_id_p1])].iterrows():
#     # Only download if necessary
#     if pd.isna(row.get("filepath")) or not row.get("filepath", False):
#         flight_code = row['flight_code']
#         url = row['url']
#         image_name = Path(row['image_name']).name
#         # Extract the filename from the URL and create a unique name using index
#         filename = downoaded_images_path.joinpath(f"{image_name}_{row['subject_id']}_{flight_code}.jpeg")
#         df_subjects.loc[index, 'filepath'] = filename
#         # Download the image
#         return_val = download_image(url, filename)
# 
#         # print(f"Downloaded {filename}")
#     if return_val == False:
#         print("there was a problem")
#         # break
        

In [40]:
df_subjects.to_csv(output_path / "iguanas-from-above-subjects_with_url.csv")